# rag-template: замеры качества

In [3]:
!nvidia-smi -L
!git clone -q https://github.com/sofetizm/rag-template.git
%cd rag-template
!pip install -q -e ".[eval]"

GPU 0: Tesla T4 (UUID: GPU-ccd10d39-041a-84d3-d0ab-bfa3c768e660)
/content/rag-template
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Installing backend dependencies ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for rag-template (pyproject.toml) ... done


Запускаем Qdrant: на больших коллекциях сервер ищет в десятки раз быстрее встроенного режима.

In [4]:
import os
import subprocess
import time
import urllib.request

if not os.path.exists("/content/qdrant"):
    !curl -sSL https://github.com/qdrant/qdrant/releases/download/v1.20.0/qdrant-x86_64-unknown-linux-gnu.tar.gz | tar xz -C /content
os.makedirs("/content/qdrant_data", exist_ok=True)
qdrant = subprocess.Popen(
    ["/content/qdrant"], cwd="/content/qdrant_data", stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL
)
for _ in range(30):
    try:
        print(urllib.request.urlopen("http://localhost:6333").read()[:80])
        break
    except Exception:
        time.sleep(1)

os.environ["RAG_QDRANT_URL"] = "http://localhost:6333"
os.environ["RAG_DEVICE"] = "cuda"

b'{"title":"qdrant - vector search engine","version":"1.20.0","commit":"222b2e44ab'


## 1. Поиск по документации FastAPI

In [5]:
!rag ingest data/demo/fastapi-ru --base-url https://fastapi.tiangolo.com/ru/ --rebuild
!rag eval retrieval --out results/golden_retrieval.json

modules.json: 100% 387/387 [00:00<00:00, 1.60MB/s]
README.md: 100% 498k/498k [00:00<00:00, 60.5MB/s]
sentence_bert_config.json: 100% 57.0/57.0 [00:00<00:00, 346kB/s]
config.json: 100% 655/655 [00:00<00:00, 2.38MB/s]

model.safetensors: downloading bytes:  63% 294M/471M [00:01<00:00, 179MB/s, 26.2MB/s  ]
model.safetensors: reconstructing file:  57% 268M/471M [00:02<00:01, 125MB/s]
model.safetensors: reconstructing file:  71% 335M/471M [00:02<00:00, 158MB/s, 25.3MB/s  ]
model.safetensors: downloading bytes: 100% 315M/315M [00:04<00:00, 68.9MB/s, 27.9MB/s  ]
model.safetensors: reconstructing file: 100% 471M/471M [00:04<00:00, 103MB/s, 37.5MB/s  ] 
Loading weights: 100% 199/199 [00:00<00:00, 23448.98it/s]
tokenizer_config.json: 100% 443/443 [00:00<00:00, 2.03MB/s]

sentencepiece.bpe.model: downloading bytes:   0% 1.38k/5.07M [00:00<26:13, 3.22kB/s]
sentencepiece.bpe.model: downloading bytes: 100% 3.62M/3.62M [00:00<00:00, 6.89MB/s,  356kB/s  ]
sentencepiece.bpe.model: reconstructing file: 

## 2. Открытые бенчмарки

In [6]:
!rag eval benchmark --dataset mteb/RuBQRetrieval --out results/bench_rubq.json
!rag eval benchmark --dataset mteb/scifact --out results/bench_scifact.json

README.md: 100% 6.23k/6.23k [00:00<00:00, 13.8MB/s]

qrels/test-00000-of-00001.parquet: downloading bytes:   0% 0.00/30.5k [00:00<?, ?B/s]
qrels/test-00000-of-00001.parquet: downloading bytes: 100% 26.8k/26.8k [00:01<00:00, 23.1kB/s, 2.59kB/s  ]
qrels/test-00000-of-00001.parquet: reconstructing file: 100% 30.5k/30.5k [00:01<00:00, 26.3kB/s, 2.96kB/s  ]
Generating test split: 100% 2845/2845 [00:00<00:00, 80204.29 examples/s]

queries/test-00000-of-00001.parquet: downloading bytes:  20% 16.9k/83.4k [00:01<00:04, 16.6kB/s]
queries/test-00000-of-00001.parquet: downloading bytes: 100% 80.4k/80.4k [00:01<00:00, 67.9kB/s, 7.79kB/s  ]
queries/test-00000-of-00001.parquet: reconstructing file: 100% 83.4k/83.4k [00:01<00:00, 70.5kB/s, 8.10kB/s  ]
Generating test split: 100% 1692/1692 [00:00<00:00, 489803.46 examples/s]

corpus/test-00000-of-00001.parquet: downloading bytes:  99% 23.8M/24.1M [00:01<00:00, 36.9MB/s, 1.67MB/s  ]
corpus/test-00000-of-00001.parquet: downloading bytes: 100% 23.8M/23.8M

## 3. Семантический кэш

In [7]:
!rag eval cache --out results/cache.json

Loading weights: 100% 199/199 [00:00<00:00, 4473.55it/s]
| Порог | Перефразы из кэша | Чужой ответ из кэша |
|---|---|---|
| 0.80 | 100% | 95% |
| 0.85 | 76% | 45% |
| 0.88 | 52% | 36% |
| 0.90 | 38% | 19% |
| 0.92 | 21% | 17% |
| 0.94 | 14% | 7% |
| 0.95 | 7% | 2% |
| 0.96 | 7% | 2% |
| 0.97 | 2% | 0% |
| 0.98 | 0% | 0% |
| 0.99 | 0% | 0% |
lowest threshold without wrong answers: 0.97
saved results/cache.json


## 4. Ответы целиком

In [8]:
from google.colab import userdata

os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY")
!rag eval answers --out results/answers.jsonl --summary results/answers_summary.json

Loading weights: 100% 199/199 [00:00<00:00, 4384.31it/s]
Loading weights: 100% 201/201 [00:00<00:00, 12122.96it/s]
hybrid+rerank: 50/50
| Конфигурация | Верных ответов | Без выдумок | Отказ, где ответ есть | Отказ, где ответа нет | Цитата из нужного документа | p50, мс | $ за 1000 ответов |
|---|---|---|---|---|---|---|---|
| bm25 | 83% (69%–92%) | 92% (80%–97%) | 7% (2%–19%) | 100% (68%–100%) | 92% (80%–97%) | 5969 | 0.13 |
| hybrid+rerank | 98% (88%–100%) | 95% (84%–99%) | 0% (0%–8%) | 100% (68%–100%) | 93% (81%–98%) | 9981 | 0.14 |
saved results/answers_summary.json


## Отчёт и скачивание

In [9]:
!python scripts/make_report.py
from IPython.display import Markdown, display

display(Markdown(open("results/RESULTS.md", encoding="utf-8").read()))
!zip -qr results.zip results
from google.colab import files

files.download("results.zip")

results/RESULTS.md


# Результаты

## RuBQ (рус., 56 тыс. абзацев)

Эмбеддинги: `intfloat/multilingual-e5-small`, реранкер: `cross-encoder/mmarco-mMiniLMv2-L12-H384-v1`, запросов: 1692.

| Конфигурация | ndcg@10 | recall@5 | recall@10 | mrr@10 | hit@1 | hit@5 |
|---|---|---|---|---|---|---|
| bm25 | 0.367 | 0.403 | 0.522 | 0.365 | 0.245 | 0.531 |
| dense | 0.680 | 0.713 | 0.803 | 0.702 | **0.603** | 0.830 |
| hybrid | 0.524 | 0.575 | 0.715 | 0.514 | 0.373 | 0.704 |
| bm25+rerank | 0.561 | 0.599 | 0.644 | 0.594 | 0.505 | 0.717 |
| dense+rerank | 0.691 | 0.729 | 0.824 | 0.702 | 0.596 | 0.840 |
| hybrid+rerank | **0.697** | **0.745** | **0.834** | **0.705** | 0.599 | **0.852** |

Разница nDCG@10 (парный бутстреп по запросам, 95% ДИ):

| Сравнение | Δ | 95% ДИ |
|---|---|---|
| dense vs bm25 | +0.313 | [+0.295; +0.332] |
| hybrid vs bm25 | +0.157 | [+0.147; +0.167] |
| bm25+rerank vs bm25 | +0.194 | [+0.180; +0.208] |
| dense+rerank vs bm25 | +0.324 | [+0.306; +0.341] |
| hybrid+rerank vs bm25 | +0.330 | [+0.314; +0.346] |
| hybrid+rerank vs hybrid | +0.173 | [+0.159; +0.187] |
| hybrid vs dense | -0.156 | [-0.171; -0.142] |

## SciFact (англ., 5 тыс. статей)

Эмбеддинги: `intfloat/multilingual-e5-small`, реранкер: `cross-encoder/mmarco-mMiniLMv2-L12-H384-v1`, запросов: 300.

| Конфигурация | ndcg@10 | recall@5 | recall@10 | mrr@10 | hit@1 | hit@5 |
|---|---|---|---|---|---|---|
| bm25 | 0.689 | 0.741 | 0.828 | 0.652 | 0.553 | 0.760 |
| dense | 0.677 | 0.734 | 0.807 | 0.645 | 0.557 | 0.757 |
| hybrid | **0.719** | **0.777** | **0.835** | **0.690** | **0.607** | **0.797** |
| bm25+rerank | 0.672 | 0.716 | 0.774 | 0.646 | 0.580 | 0.737 |
| dense+rerank | 0.676 | 0.719 | 0.782 | 0.650 | 0.580 | 0.743 |
| hybrid+rerank | 0.669 | 0.719 | 0.774 | 0.642 | 0.570 | 0.740 |

Разница nDCG@10 (парный бутстреп по запросам, 95% ДИ):

| Сравнение | Δ | 95% ДИ |
|---|---|---|
| dense vs bm25 | -0.012 | [-0.047; +0.023] |
| hybrid vs bm25 | +0.030 | [+0.010; +0.049] |
| bm25+rerank vs bm25 | -0.018 | [-0.047; +0.012] |
| dense+rerank vs bm25 | -0.013 | [-0.043; +0.016] |
| hybrid+rerank vs bm25 | -0.020 | [-0.049; +0.009] |
| hybrid+rerank vs hybrid | -0.050 | [-0.077; -0.023] |
| hybrid vs dense | +0.042 | [+0.016; +0.069] |

## Документация FastAPI (рус., 42 вопроса)

Эмбеддинги: `intfloat/multilingual-e5-small`, реранкер: `cross-encoder/mmarco-mMiniLMv2-L12-H384-v1`, запросов: 84.

| Конфигурация | ndcg@10 | recall@5 | recall@10 | mrr@10 | hit@1 | hit@5 |
|---|---|---|---|---|---|---|
| bm25 | 0.810 | 0.940 | 0.976 | 0.755 | 0.595 | 0.940 |
| dense | 0.852 | 0.964 | 0.976 | 0.811 | 0.702 | 0.964 |
| hybrid | 0.860 | 0.964 | **0.988** | 0.817 | 0.702 | 0.964 |
| bm25+rerank | 0.890 | **0.988** | **0.988** | 0.858 | **0.774** | **0.988** |
| dense+rerank | **0.893** | **0.988** | **0.988** | **0.861** | 0.762 | **0.988** |
| hybrid+rerank | 0.887 | **0.988** | **0.988** | 0.853 | 0.750 | **0.988** |

Разница nDCG@10 (парный бутстреп по запросам, 95% ДИ):

| Сравнение | Δ | 95% ДИ |
|---|---|---|
| dense vs bm25 | +0.042 | [-0.014; +0.099] |
| hybrid vs bm25 | +0.050 | [+0.008; +0.093] |
| bm25+rerank vs bm25 | +0.080 | [+0.023; +0.139] |
| dense+rerank vs bm25 | +0.083 | [+0.025; +0.142] |
| hybrid+rerank vs bm25 | +0.077 | [+0.019; +0.137] |
| hybrid+rerank vs hybrid | +0.027 | [-0.026; +0.082] |
| hybrid vs dense | +0.008 | [-0.032; +0.045] |

![nDCG@10](retrieval.png)

## Семантический кэш

Эмбеддинги: `intfloat/multilingual-e5-small`.

| Порог | Перефразы из кэша | Чужой ответ из кэша |
|---|---|---|
| 0.80 | 100% | 95% |
| 0.85 | 76% | 45% |
| 0.88 | 52% | 36% |
| 0.90 | 38% | 19% |
| 0.92 | 21% | 17% |
| 0.94 | 14% | 7% |
| 0.95 | 7% | 2% |
| 0.96 | 7% | 2% |
| 0.97 | 2% | 0% |
| 0.98 | 0% | 0% |
| 0.99 | 0% | 0% |

Минимальный порог без чужих ответов: 0.97.

![Кэш](cache.png)

## Ответы целиком (LLM-судья)

| Конфигурация | Верных ответов | Без выдумок | Отказ, где ответ есть | Отказ, где ответа нет | Цитата из нужного документа | p50, мс | $ за 1000 ответов |
|---|---|---|---|---|---|---|---|
| bm25 | 83% (69%–92%) | 92% (80%–97%) | 7% (2%–19%) | 100% (68%–100%) | 92% (80%–97%) | 5969 | 0.13 |
| hybrid+rerank | 98% (88%–100%) | 95% (84%–99%) | 0% (0%–8%) | 100% (68%–100%) | 93% (81%–98%) | 9981 | 0.14 |


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>